In [1]:
#Part 2
#Task 1
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms

# Transforms, datasets, and dataloaders
transform = transforms.Compose([transforms.ToTensor()])
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

mlp = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

def evaluate(model, loader):c:\\Users\\kaush\\Downloads\\ECE588_fa2026_Lab01\\ECE588_fa2026_Lab01\\Part02\\ece588_part2_question.ipynb$0
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            pred = outputs.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    return correct / total

# Training loop for 3 epochs
for epoch in range(3):
    mlp.train()
    running_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad()
        outputs = mlp(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_acc = evaluate(mlp, trainloader)
    test_acc = evaluate(mlp, testloader)
    print(f"Epoch {epoch+1}: loss={running_loss/len(trainloader):.4f}  "
          f"train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 15.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 476kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.13MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.62MB/s]


Epoch 1: loss=0.2865  train_acc=0.9659  test_acc=0.9604
Epoch 2: loss=0.1090  train_acc=0.9796  test_acc=0.9703
Epoch 3: loss=0.0720  train_acc=0.9771  test_acc=0.9660


In [2]:
#Task 2
class AlexNetMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define convolutional layers
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        # TODO: define fully connected classifier
        self.classifier = nn.Sequential(
            nn.Linear(128*7*7, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

alexnet = AlexNetMNIST()
optimizer = optim.Adam(alexnet.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# TODO: training loop
for epoch in range(3):
    alexnet.train()
    running_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad()
        outputs = alexnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_acc = evaluate(alexnet, trainloader)
    test_acc = evaluate(alexnet, testloader)
    print(f"Epoch {epoch+1}: loss={running_loss/len(trainloader):.4f}  "
          f"train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")

Epoch 1: loss=0.1376  train_acc=0.9893  test_acc=0.9880
Epoch 2: loss=0.0426  train_acc=0.9936  test_acc=0.9902
Epoch 3: loss=0.0281  train_acc=0.9946  test_acc=0.9898


In [3]:
#Task 3
import numpy as np
import torch.nn.functional as F
def conv2d(image, kernel):
    H, W = image.shape
    kH, kW = kernel.shape
    out = np.zeros((H-kH+1, W-kW+1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i,j] = np.sum(image[i:i+kH, j:j+kW] * kernel)
    return out

img = trainset[0][0][0].numpy()
kernel = np.array([[1,0,-1],[1,0,-1],[1,0,-1]])
out = conv2d(img, kernel)
print(out.shape)

img_t = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
kernel_t = torch.tensor(kernel, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
torch_out = F.conv2d(img_t, kernel_t).squeeze().numpy()

print("Manual output shape:", out.shape)
print("PyTorch output shape:", torch_out.shape)
print("Max difference:", np.max(np.abs(out - torch_out)))
print("Match:", np.allclose(out, torch_out, atol=1e-5))

(26, 26)
Manual output shape: (26, 26)
PyTorch output shape: (26, 26)
Max difference: 2.076849341392517e-07
Match: True
